# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² Mental Health Survey Dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# The metadata attribute is a single object
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's list all record sets in the dataset with their `@id` and fields (with field `@id`s), using the `mlcroissant` API.

In [ ]:
# Get all record sets from the dataset
record_sets_info = []
for rs in dataset.record_sets:
    rs_id = rs.id  # @id
    rs_name = rs.name
    fields = getattr(rs, 'fields', [])
    field_ids = [field.id for field in fields]  # fields' @id
    record_sets_info.append({'id': rs_id, 'name': rs_name, 'fields': field_ids})
    print(f"RecordSet: {rs_name} (@id={rs_id})")
    print(f"  Fields: {field_ids}")
    print()

# List column information for each RecordSet if available
for rs in dataset.record_sets:
    columns = getattr(rs, 'columns', [])
    if columns:
        column_ids = [col.id for col in columns]
        print(f"RecordSet '{rs.name}' columns: {column_ids}")
    else:
        print(f"RecordSet '{rs.name}' has no explicit columns listed.")
    print()

## 3. Data Extraction
Load data from all record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

We will extract the data for each `RecordSet` using its `@id`.

In [ ]:
# Extract data from each record set by their @id
dataframes = {}
record_set_ids = [rs['id'] for rs in record_sets_info]
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from RecordSet @id={record_set_id}")
    except Exception as e:
        print(f"Could not load records from RecordSet @id={record_set_id}: {e}")

# For demonstration, print the columns from the first RecordSet with data
for rs_id, df in dataframes.items():
    if not df.empty:
        print(f"Columns in DataFrame loaded from RecordSet @id={rs_id}:")
        print(df.columns.tolist())
        print(df.head())
        break

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's pick a numeric field (`@id`) for analysis. We'll filter for high scores (e.g., GAD-7 scale), normalize, and group by a demographic field.

In [ ]:
# Example: GAD-7 scores and grouping by 'village' from main survey record set
# Please update these @id values based on the actual field IDs found in your record set overview

# Find the main survey RecordSet (by guessing typical naming or using the first non-empty loaded DataFrame)
main_rs_id = None
for rs_id, df in dataframes.items():
    if not df.empty:
        main_rs_id = rs_id
        break

df = dataframes[main_rs_id]

# Choose numeric field and group field; replace with actual @id as needed
numeric_field_id = None
group_field_id = None

# Try to find GAD-7 or PHQ-9 fields among DataFrame columns using substring matching
for col in df.columns:
    if 'GAD-7' in col or 'gad' in col.lower():
        numeric_field_id = col
    if 'village' in col.lower():
        group_field_id = col

# Fallback: use PHQ-9 or PSQ fields
if numeric_field_id is None:
    for col in df.columns:
        if 'PHQ-9' in col or 'phq' in col.lower():
            numeric_field_id = col
        if 'psq' in col.lower():
            numeric_field_id = col

if numeric_field_id is None:
    # default to first number column
    for col in df.columns:
        if df[col].dtype in ['int64', 'float64']:
            numeric_field_id = col
            break

# Fallback: use first categorical string column for grouping
if group_field_id is None:
    for col in df.columns:
        if df[col].dtype == 'object':
            group_field_id = col
            break

threshold = 10
if numeric_field_id is not None and numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with '{numeric_field_id}' > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized '{numeric_field_id}' for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by the group field
    if group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by '{group_field_id}':")
        print(grouped_df.head())
else:
    print("Could not find a numeric field for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we display histograms and group means for the selected numeric field.

In [ ]:
if numeric_field_id is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    df[numeric_field_id].hist(bins=20)
    plt.title(f"Distribution of '{numeric_field_id}' Scores")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id is not None and group_field_id in df.columns:
        # Barplot of group means
        grouped_df = df.groupby(group_field_id)[numeric_field_id].mean()
        grouped_df.plot(kind='bar', figsize=(8, 4))
        plt.title(f"Mean '{numeric_field_id}' by '{group_field_id}'")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()

## 6. Conclusion
This notebook demonstrated loading, overview, extraction, and analysis of the FAIR² Mental Health Survey Dataset using the `mlcroissant` library.

Key findings include an overview of available record sets and fields, basic exploratory data analysis (with normalization and grouping), and visualization of mental health assessment results.

For further analysis, consult the dataset's Croissant schema for additional field `@id`s and enrich your analysis with additional EDA and reporting.